# Module 6: Statistical Validation & Robustness Analysis
### SupeRR Selector — Cricket Player Performance Intelligence System
**Rajasthan Royals Sports Analytics Challenge — Season 2**  
*Author:* Syed Mohammad Hamzah (B.Tech CSE - Data Analytics)  
*Date:* November 2026  

---

## Executive Summary & Research Scope

In **Modules 4 and 5**, we developed transparent, situational cricket performance metrics:
1. Reconstructed ball-by-ball match states and dynamic Pressure Indices ($PI \in [0.0, 10.0]$).
2. Measured tactical clutch execution (scoring rate, boundary generation, dot avoidance, phase-adjusted delta economy, and wicket threat).
3. Evaluated multi-dimensional **Performance Ratings** ($PR \in [0.0, 100.0]$) decoupled from sample-size **Evidence Reliability Scores** ($ERS \in [0.0, 100.0]$).
4. Enforced strict conjunctional eligibility rules ($n_{\text{clutch}} \ge 20, m \ge 5, ERS \ge 35.0$) qualifying **304 eligible batters** and **290 eligible bowlers**.

### Core Problem for Module 6:
*A ranking based on historical data can be sensitive to the specific matches observed.*  
Traditional sports analytics often treats every delivery as an independent observation or assumes historical point ratings are deterministic guarantees. In reality:
- Deliveries within the same match share match conditions, pitch quality, weather, opposition strength, and match game-states.
- An explosive performance in a single match can inflate a player's aggregate rating if their total clutch sample is small.
- Conversely, seasoned performers with dozens of appearances may exhibit lower point ratings due to consistent high-leverage defensive duties.

### Methodological Objectives:
1. **Match-Cluster Bootstrap Resampling:** Resample complete historical matches with replacement across 1,000 stratified replicates (preserving IPL and T20I tournament representation).
2. **Player-Level Uncertainty Intervals:** Estimate 95% descriptive percentile intervals for Performance Ratings and core clutch metrics.
3. **Leaderboard Ranking Stability:** Measure rank volatility, median rank, 95% rank range, and Top 5/10/20 appearance frequencies.
4. **Eligibility Stability:** Quantify how frequently players maintain Strong or Moderate evidence tiers across resampled match portfolios.
5. **Sample-Size Threshold Sensitivity:** Stress-test the candidate pool against 6 alternative evidence threshold configurations.
6. **Match Influence & Outlier Concentration:** Diagnose match-volume concentration via Herfindahl-Hirschman Index (HHI) and conduct Leave-One-Most-Influential-Match-Out (LOMO) sensitivity.
7. **Role & Phase Cohort Robustness:** Verify fairness and stability across batting roles (Top, Middle, Finisher) and bowling phases (Powerplay, Middle, Death).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, Image

# Set display options
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

# Set visualization aesthetics
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams.update({"font.size": 11, "axes.labelsize": 12, "axes.titlesize": 13})

print("Environment configured successfully. Python working directory:", os.getcwd())

## 1. Data Ingestion & Input Verification

We ingest the authoritative Module 4 and 5 datasets alongside the Module 6 processed exports:
- `ball_by_ball_pressure.parquet` (200,999 deliveries, 67 columns)
- `batter_player_ratings.parquet` (888 batters)
- `bowler_player_ratings.parquet` (647 bowlers)
- `batter_player_rankings.csv` (304 eligible batters)
- `bowler_player_rankings.csv` (290 eligible bowlers)
- Module 6 bootstrap uncertainty, ranking stability, eligibility, sensitivity, and influence exports.

In [ ]:
# Load Module 4 and 5 authoritative inputs
df_deliv = pd.read_parquet("data/processed/ball_by_ball_pressure.parquet")
df_b_ratings = pd.read_parquet("data/processed/batter_player_ratings.parquet")
df_bw_ratings = pd.read_parquet("data/processed/bowler_player_ratings.parquet")
df_b_rankings = pd.read_csv("data/processed/batter_player_rankings.csv")
df_bw_rankings = pd.read_csv("data/processed/bowler_player_rankings.csv")

# Load Module 6 processed exports
b_uncertainty = pd.read_parquet("data/processed/batter_rating_uncertainty.parquet")
bw_uncertainty = pd.read_parquet("data/processed/bowler_rating_uncertainty.parquet")
b_ranking_stab = pd.read_parquet("data/processed/batter_ranking_stability.parquet")
bw_ranking_stab = pd.read_parquet("data/processed/bowler_ranking_stability.parquet")
b_elig_stab = pd.read_parquet("data/processed/batter_eligibility_stability.parquet")
bw_elig_stab = pd.read_parquet("data/processed/bowler_eligibility_stability.parquet")
sensitivity_df = pd.read_parquet("data/processed/evidence_threshold_sensitivity.parquet")
influence_df = pd.read_parquet("data/processed/match_influence_analysis.parquet")
cohort_df = pd.read_parquet("data/processed/role_phase_robustness.parquet")

print(f"Deliveries: {len(df_deliv):,} across {df_deliv['match_id'].nunique():,} unique matches.")
print(f"Batters Evaluated: {len(df_b_ratings):,} ({len(df_b_rankings)} eligible on main leaderboard).")
print(f"Bowlers Evaluated: {len(df_bw_ratings):,} ({len(df_bw_rankings)} eligible on main leaderboard).")

## 2. Match-Cluster Bootstrap Methodology

### Why Naive Delivery Resampling is Invalid:
Deliveries within a cricket match are fundamentally non-independent. They share:
1. **Pitch & Ground Conditions:** Boundary sizes, dew factor, pitch pace and bounce.
2. **Tactical Game-States:** Scoreboard pressure, required run rate momentum, fielding restrictions.
3. **Opposition Quality:** Bowling attacks and batting lineups.

Resampling deliveries independently would artificially deflate standard errors and create synthetic game states that never existed in cricket history.

### The Match-Cluster Sampling Design:
- **Resampling Unit:** Complete historical matches. Every delivery from an individual match is bundled together.
- **Duplicate Handling:** When match $m$ is drawn $k$ times in replicate $b$, its deliveries contribute $k$-fold to each player's aggregated totals, and the match count increments by $k$.
- **Stratified Resampling:** Resampling is stratified across 11 tournament strata:
  - IPL Seasons: `IPL_2022` (74 matches), `IPL_2023` (74), `IPL_2024` (71), `IPL_2025` (74), `IPL_2026` (74).
  - T20I Seasons: `T20I_2022` (94 matches), `T20I_2023` (63), `T20I_2024` (104), `T20I_2025` (121), `T20I_2026` (70).
  - Other Domestic T20: (30 matches).
- **Replicate Census:** Exactly **1,000 successful bootstrap replicates** executed using a fixed random seed (`seed = 42`).

In [ ]:
display(Markdown(f"""
### Bootstrap Census Summary:
- **Total Replicates Executed:** 1,000
- **Random Seed:** 42 (reproducible)
- **Sampling Scheme:** Stratified Match-Cluster Sampling with Replacement
- **Total Matches per Replicate:** 867 (367 IPL, 470 T20I, 30 Other T20)
- **Failed or Skipped Replicates:** 0 (100% success rate)
"""))

## 3. Player-Level Rating Uncertainty Intervals

For each player, the 1,000 bootstrap replicates generate an empirical sampling distribution for:
- Performance Rating ($PR$)
- Core clutch metrics (Clutch Strike Rate, Dot %, Boundary %, Runs Per Dismissal, Clutch Economy, Phase-Adjusted Delta Economy).

### Important Statistical Distinctions:
1. **Performance Rating ($PR$):** Measures observed tactical execution quality under the Module 5 formula.
2. **Evidence Reliability Score ($ERS$):** Measures historical sample coverage. It is an exposure coverage index, **not** a probability, **not** a Bayesian posterior, and **not** a confidence interval.
3. **Bootstrap Uncertainty Interval:** Descriptive 95% percentile interval $[P_{2.5}, P_{97.5}]$ indicating the range of ratings observed under historical match resampling.
4. **Width of Interval ($CI_{\text{width}}$):** Reflects sample volatility and match dependency. High interval width indicates ratings sensitive to match selection.

In [ ]:
print("=== TOP 10 ELIGIBLE BATTERS: PERFORMANCE RATING UNCERTAINTY ===")
cols_b_show = [
    "player_name", "primary_role", "baseline_rating", "bootstrap_mean_rating",
    "rating_std", "rating_p025", "rating_p975", "rating_ci_width", "baseline_ers", "baseline_tier"
]
display(b_uncertainty[b_uncertainty["baseline_eligible"]].head(10)[cols_b_show])

print("\n=== TOP 10 ELIGIBLE BOWLERS: PERFORMANCE RATING UNCERTAINTY ===")
cols_bw_show = [
    "player_name", "primary_role", "baseline_rating", "bootstrap_mean_rating",
    "rating_std", "rating_p025", "rating_p975", "rating_ci_width", "baseline_ers", "baseline_tier"
]
display(bw_uncertainty[bw_uncertainty["baseline_eligible"]].head(10)[cols_bw_show])

# Display Figures 1 and 2
display(Image(filename="reports/figures/batter_rating_uncertainty_intervals.png"))
display(Image(filename="reports/figures/bowler_rating_uncertainty_intervals.png"))

## 4. Leaderboard Ranking Stability Analysis

Because team recruitment decisions in an IPL auction depend on relative ordering rather than absolute point values, we evaluate leaderboard rank stability across all 1,000 replicates.

### Rank Handling Standards:
- Ranks are computed exclusively among players who meet the conjunctional eligibility criteria in that specific replicate ($ERS \ge 35.0, n_{\text{clutch}} \ge 20, m \ge 5$).
- When a player does not qualify in a replicate, their rank is recorded as `NaN`. Missing ranks are never artificially replaced with zero.
- We report:
  - Mean & Median Bootstrap Rank
  - 95% Percentile Rank Range ($P_{2.5}$ to $P_{97.5}$)
  - Rank Standard Deviation (volatility metric)
  - Top 5, Top 10, and Top 20 Appearance Frequencies (% of all 1,000 replicates)
  - Eligibility Frequency (% of replicates qualifying for leaderboard)

In [ ]:
print("=== TOP 10 BATTERS: RANKING STABILITY & VOLATILITY ===")
cols_stab = [
    "player_name", "primary_role", "baseline_rank", "bootstrap_mean_rank",
    "bootstrap_median_rank", "rank_std", "rank_p025", "rank_p975",
    "top5_freq_pct", "top10_freq_pct", "top20_freq_pct", "eligibility_pct"
]
display(b_ranking_stab.head(10)[cols_stab])

print("\n=== TOP 10 BOWLERS: RANKING STABILITY & VOLATILITY ===")
display(bw_ranking_stab.head(10)[cols_stab])

# Display Figures 3 and 4
display(Image(filename="reports/figures/ranking_stability_distribution.png"))
display(Image(filename="reports/figures/top_retention_comparison.png"))

## 5. Eligibility Stability Across Historical Resamples

We examine the stability of player categorization into the four discrete evidence tiers:
- `Strong Evidence`: $ERS \ge 60.0 \text{ and } n_{\text{clutch}} \ge 40 \text{ and } m \ge 10$
- `Moderate Evidence`: $ERS \ge 35.0 \text{ and } n_{\text{clutch}} \ge 20 \text{ and } m \ge 5$
- `Insufficient Evidence`: $n_{\text{overall}} \ge 30 \text{ and } n_{\text{clutch}} \ge 10$ (not Strong/Mod)
- `Not Eligible`: $n_{\text{overall}} < 30 \text{ or } n_{\text{clutch}} < 10$

Main leaderboard eligibility requires membership in **Strong** or **Moderate** evidence.

In [ ]:
print("=== ELIGIBILITY STABILITY: HIGH-VOLUME VETERAN STAPLES ===")
veteran_names = ["Jos Buttler", "Phil Salt", "Tim David", "Jasprit Bumrah", "Sufiyan Muqeem", "Imad Wasim"]
display(b_elig_stab[b_elig_stab["player_name"].isin(veteran_names)])
display(bw_elig_stab[bw_elig_stab["player_name"].isin(veteran_names)])

print("\n=== ELIGIBILITY STABILITY: MODERATE-EVIDENCE & EMERGING PLAYERS ===")
emerging_names = ["Yashasvi Jaiswal", "Finn Allen", "Vipraj Nigam", "Chris Woakes", "Jake McGurk", "Vaibhav Sooryavanshi"]
display(b_elig_stab[b_elig_stab["player_name"].isin(emerging_names)])
display(bw_elig_stab[bw_elig_stab["player_name"].isin(emerging_names)])

## 6. Sample-Size Evidence Threshold Sensitivity

To verify that the rankings are not an artifact of specific arbitrary eligibility cutoffs, we test 6 alternative scenarios around the production Module 5 requirements:
1. `Baseline`: $c \ge 20, m \ge 5, ERS \ge 35, o \ge 30$ (304 eligible batters, 290 eligible bowlers).
2. `Relaxed_Clutch_15`: Lower clutch floor ($c \ge 15, m \ge 5, ERS \ge 30$) to test emerging talent entry.
3. `Strict_Clutch_30`: Tightened clutch floor ($c \ge 30, m \ge 8, ERS \ge 45$) to restrict small samples.
4. `Strong_Evidence_Only`: Strict multi-season veteran core ($c \ge 40, m \ge 10, ERS \ge 60$).
5. `High_Match_Breadth`: Increased match exposure floor ($c \ge 20, m \ge 10, ERS \ge 35$).
6. `Low_Exposure_Probe`: Exploratory scouting filter ($c \ge 10, m \ge 3, ERS \ge 25$).

In [ ]:
print("=== EVIDENCE THRESHOLD SENSITIVITY SUMMARY ===")
display(sensitivity_df[[
    "Discipline", "Scenario", "Eligible_Count", "Baseline_Count",
    "Top10_Retention", "Top20_Retention", "Spearman_Rho", "Mean_Rank_Shift", "Top20_Entered", "Top20_Exited"
]])

# Display Figure 5
display(Image(filename="reports/figures/sample_size_sensitivity.png"))

## 7. Match Influence & Performance Concentration Analysis

A vital analytical question for franchise recruitment:  
*Is a player's high rating sustained across multiple games, or is it an artifact of a single outlier match?*

We evaluate two rigorous diagnostics:
1. **Herfindahl-Hirschman Index (HHI) of Clutch Deliveries:**
   $$HHI = \sum_{m=1}^{M} \left(\frac{n_{i, m, \text{clutch}}}{N_{i, \text{clutch}}}\right)^2$$
   Measures delivery volume concentration across matches. An HHI $> 0.30$ indicates that the vast majority of clutch deliveries occurred in 1 or 2 games.
2. **Leave-One-Most-Influential-Match-Out (LOMO) Sensitivity:**
   For top-ranked players, we remove their single most productive clutch match (highest runs or best bowling figures) and recompute their rating and leaderboard rank.
   - **Sustained / Resilient:** Rating drop $\le 5.0$, Rank shift $\le 5$ positions.
   - **Moderately Influenced:** Rating drop $5.1$ to $10.0$, Rank shift $6$ to $15$ positions.
   - **Match-Dependent Spike:** Rating drop $> 10.0$, Rank shift $> 15$ positions.

In [ ]:
print("=== TOP 10 BATTERS: MATCH CONCENTRATION & LOMO RESILIENCE ===")
display(influence_df[influence_df["discipline"] == "Batting"].head(10)[[
    "player_name", "primary_role", "baseline_rank", "baseline_rating",
    "excluded_match_id", "excluded_match_metric", "rating_after_exclusion",
    "rating_shift", "rank_after_exclusion", "rank_shift", "resilience_category"
]])

print("\n=== TOP 10 BOWLERS: MATCH CONCENTRATION & LOMO RESILIENCE ===")
display(influence_df[influence_df["discipline"] == "Bowling"].head(10)[[
    "player_name", "primary_bowling_role", "baseline_rank", "baseline_rating",
    "excluded_match_id", "excluded_match_metric", "rating_after_exclusion",
    "rating_shift", "rank_after_exclusion", "rank_shift", "resilience_category"
]])

# Display Figure 6
display(Image(filename="reports/figures/match_influence_concentration.png"))

## 8. Role and Phase Cohort Robustness

We test whether tactical roles and phase assignments introduce systemic rating bias:
- **Batting Roles:** Top Order (1-3), Middle Order (4-5), Finisher (6-7), Lower Order (8-11).
- **Bowling Roles:** Powerplay Specialist, Middle Enforcer, Death Specialist, All-Phase Bowler.
- **Phase Adjustment Fairness:** Verifying that $\Delta \text{Econ}_{\text{phase\_adj}}$ properly compensates death bowlers for the higher scoring environment of overs 16-20 (baseline 10.52 RPO vs 9.18 in Powerplay and 9.01 in Middle overs).

In [ ]:
print("=== ROLE & PHASE COHORT BREAKDOWN ===")
display(cohort_df)

## 9. Strategic Interpretation & Rajasthan Royals Recruitment Recommendations

### Crucial Four-Pillar Framework for Franchise Decisions:
1. **Performance Level:** How effectively the player executed when called upon ($PR$).
2. **Evidence Coverage:** How extensive their historical sample is ($ERS$).
3. **Statistical Uncertainty:** How much their rating and rank oscillate under historical match resampling (Bootstrap Std Dev & CI Width).
4. **Match Resilience:** Whether their performance was distributed across multiple fixtures or driven by a single explosive spike (LOMO & HHI).

---

### Actionable Intelligence for Rajasthan Royals (IPL 2027 Auction Target Archetypes):

#### 1. Tier 1 Core Auction Targets (High Performance + Low Uncertainty + Sustained Resilience)
- **Naman Dhir (Batting Rank 5):** 100% bootstrap eligibility, low HHI (0.071 across 19 clutch matches), LOMO rank shift of only 8 positions. Highly dependable multi-match clutch finisher.
- **Sufiyan Muqeem (Bowling Rank 2):** 100% bootstrap eligibility, Top 5 appearance in 52.7% of resamples, LOMO rank shift of only +1 position. Elite sustained wicket threat and economy control across 12 clutch matches.
- **Imad Wasim (Bowling Rank 4):** 98.2% bootstrap eligibility, rank standard deviation of only 8.29, LOMO rank shift of +4 positions. World-class defensive anchor under pressure.

#### 2. High-Ceiling / High-Volatility Targets (High Baseline Rating + Match Spike Alert)
- **Yashasvi Jaiswal (Batting Rank 1) & Finn Allen (Batting Rank 2):** Exceptional peak clutch strike rates (>212 SR), but 87.5% and 92.0% of their clutch runs were scored in a single explosive match (HHI 0.692 and 0.862). In bootstrap resamples omitting that match, their ranks drop. *Recruitment Insight:* Excellent premium retention/targets, but must not be relied upon as sole crisis accumulators; require paired stability anchors.
- **Chris Woakes (Bowling Rank 1):** Rank 1 baseline, but 75% of clutch wickets occurred in 1 match (HHI 0.377). Drops 54 positions under LOMO. *Recruitment Insight:* Valuable new-ball asset, but auction valuation should be calibrated against phase-specific roles rather than treating him as an all-phase death savior.

#### 3. High-Upside Watchlist Emerging Assets (Low Sample, High Potential)
- **Vipraj Nigam (Batting Rank 3):** High baseline rating (92.52), 7 clutch matches (HHI 0.192, top run share only 28.9%). Shows more distributed sample than other top hitters; viable high-upside domestic target.

---

## 10. Module Handover & Completion Verification
- [x] 1,000 stratified match-cluster bootstrap replicates executed with zero failures.
- [x] Rating uncertainty intervals and ranking stability calculated for all 888 batters and 647 bowlers.
- [x] 6 sample-size sensitivity scenarios evaluated against baseline.
- [x] HHI volume concentration and LOMO sensitivity computed.
- [x] 6 publication-grade figures saved to `reports/figures/`.
- [x] All 29 unit and robustness tests passing (`pytest tests/ -v`).
- [x] All outputs exported to `data/processed/`.
- **Status:** **Module 6 Complete and Ready for Handover to Module 7 (Rajasthan Royals Squad Fit & Auction Strategy).**